# Show do Milhão - Python (Google Colab)
Integrado ao Supabase com persistência online de partidas e ranking.

In [ ]:
# 1. Instalação das dependências necessárias no Colab
!pip install requests python-dotenv

In [ ]:
# 2. Configurações do Supabase
SUPABASE_URL = "https://wfukynvpqnbymapzwjwe.supabase.co"
SUPABASE_KEY = "sb_publishable_CsxbQBtDR4FgBqiW0FG8JQ_KIWMMwCt"
NOME_JOGO = "Show do Milhão - Python"
TIMEOUT = 10

PREMIOS = [0, 1000, 5000, 10000, 25000, 50000, 100000, 200000, 500000, 1000000]
ESTRUTURA = [("facil", 3), ("media", 3), ("dificil", 3)]
LETRAS = ["a", "b", "c", "d"]
partidas_pendentes = []

In [ ]:
# 3. Código do Jogo
import random
import time
import uuid
from datetime import datetime, timezone
import requests

class ErroBanco(Exception):
    def __init__(self, mensagem, status=None):
        super().__init__(mensagem)
        self.status = status

def requisitar(metodo, caminho, **kwargs):
    try:
        resposta = requests.request(
            metodo,
            f"{SUPABASE_URL}/rest/v1/{caminho}",
            headers={"apikey": SUPABASE_KEY, "Content-Type": "application/json"},
            timeout=TIMEOUT,
            **kwargs,
        )
    except requests.RequestException:
        raise ErroBanco("Sem conexão com o banco de dados.")
    if resposta.status_code >= 400:
        raise ErroBanco(f"O banco recusou o pedido (código {resposta.status_code}).", status=resposta.status_code)
    return resposta

def buscar_jogo_id():
    resposta = requisitar("GET", "jogos", params={"select": "id", "nome": f"eq.{NOME_JOGO}"})
    jogos = resposta.json()
    return jogos[0]["id"] if jogos else None

def sortear_perguntas(todas):
    sorteadas = []
    for dificuldade, quantidade in ESTRUTURA:
        da_dificuldade = [p for p in todas if p["dificuldade"] == dificuldade]
        if len(da_dificuldade) < quantidade:
            print(f"Aviso: só há {len(da_dificuldade)} pergunta(s) de nível '{dificuldade}'.")
        sorteadas += random.sample(da_dificuldade, min(quantidade, len(da_dificuldade)))
    return sorteadas

def buscar_perguntas():
    colunas = "id,enunciado,alternativa_a,alternativa_b,alternativa_c,alternativa_d,dificuldade"
    resposta = requisitar("GET", "perguntas", params={"select": colunas})
    return sortear_perguntas(resposta.json())

def verificar_resposta(pergunta_id, letra):
    resposta = requisitar("POST", "rpc/verificar_resposta", json={"p_id": pergunta_id, "p_resposta": letra})
    return resposta.json() is True

def calcular_premio(acertos):
    acertos = max(0, min(acertos, len(PREMIOS) - 1))
    return PREMIOS[acertos]

def formatar_dinheiro(valor):
    texto = f"{valor:,.2f}"
    return "R$ " + texto.replace(",", "X").replace(".", ",").replace("X", ".")

def formatar_tempo(segundos):
    total = int(round(float(segundos)))
    return f"{total // 60} min {total % 60:02d} s"

def formatar_data(texto_iso):
    return datetime.fromisoformat(texto_iso).astimezone().strftime("%d/%m/%Y %H:%M")

def pedir_apelido():
    while True:
        apelido = input("Digite seu apelido: ").strip()
        if not apelido:
            print("O apelido não pode ser vazio. Tente de novo.")
        elif len(apelido) > 20:
            print("O apelido pode ter no máximo 20 caracteres. Tente de novo.")
        else:
            return apelido

def pedir_resposta():
    while True:
        escolha = input("Sua resposta [A/B/C/D] ou [P] para parar: ").strip().lower()
        if escolha in LETRAS or escolha == "p":
            return escolha
        print("Entrada inválida. Digite A, B, C, D ou P.")

def perguntar_sim_nao(texto):
    while True:
        escolha = input(f"{texto} [S/N]: ").strip().lower()
        if escolha in ("s", "n"):
            return escolha == "s"
        print("Digite S ou N.")

def mostrar_pergunta(numero, total, pergunta):
    premio_acertar = formatar_dinheiro(calcular_premio(numero))
    premio_parar = formatar_dinheiro(calcular_premio(numero - 1))
    print(f"\nPergunta {numero}/{total} ({pergunta['dificuldade']}) - vale {premio_acertar}")
    print(pergunta["enunciado"])
    for letra in LETRAS:
        print(f"  {letra.upper()}) {pergunta['alternativa_' + letra]}")
    print(f"(Se parar agora, você leva {premio_parar})")

def conferir_resposta(pergunta_id, letra):
    gasto = 0.0
    while True:
        inicio = time.perf_counter()
        try:
            acertou = verificar_resposta(pergunta_id, letra)
            return acertou, gasto + time.perf_counter() - inicio
        except ErroBanco as erro:
            gasto += time.perf_counter() - inicio
            print(f"Falha ao conferir a resposta: {erro}")
            if not perguntar_sim_nao("Tentar novamente?"):
                return None, gasto

def salvar_partida(partida):
    while True:
        try:
            requisitar("POST", "partidas", json=partida)
            salvo = True
        except ErroBanco as erro:
            salvo = erro.status == 409
            if not salvo:
                print(f"Falha ao salvar o resultado: {erro}")
        if salvo:
            print("Resultado salvo com sucesso!")
            if partida in partidas_pendentes:
                partidas_pendentes.remove(partida)
            return True
        if partida not in partidas_pendentes:
            partidas_pendentes.append(partida)
        if not perguntar_sim_nao("Tentar enviar novamente?"):
            print("Resultado mantido na memória.")
            return False

def jogar():
    apelido = pedir_apelido()
    try:
        jogo_id = buscar_jogo_id()
        perguntas = buscar_perguntas()
    except ErroBanco as erro:
        print(f"Não foi possível iniciar: {erro}")
        return
    if jogo_id is None:
        print(f"O jogo '{NOME_JOGO}' não existe no banco.")
        return
    if not perguntas:
        print("O banco não tem perguntas cadastradas.")
        return

    partida_id = str(uuid.uuid4())
    acertos = 0
    motivo = "vitoria"
    tempo_banco = 0.0
    inicio = time.perf_counter()

    for numero, pergunta in enumerate(perguntas, start=1):
        mostrar_pergunta(numero, len(perguntas), pergunta)
        escolha = pedir_resposta()
        if escolha == "p":
            motivo = "desistencia"
            break
        acertou, gasto = conferir_resposta(pergunta["id"], escolha)
        tempo_banco += gasto
        if acertou is None:
            print("Partida cancelada por falha de conexão.")
            return
        if acertou:
            acertos += 1
            print(f"Correto! Você ganhou {formatar_dinheiro(calcular_premio(acertos))}.")
        else:
            motivo = "erro"
            print("Resposta errada!")
            break

    tempo = round(time.perf_counter() - inicio - tempo_banco, 2)
    valor = calcular_premio(acertos)
    textos = {"erro": "errou uma pergunta", "desistencia": "desistiu", "vitoria": "acertou todas!"}
    print("\n===== FIM DA PARTIDA =====")
    print(f"Jogador: {apelido} | Motivo: {textos[motivo]}")
    print(f"Acertos: {acertos} | Prêmio: {formatar_dinheiro(valor)} | Tempo: {formatar_tempo(tempo)}")

    salvar_partida({
        "id": partida_id,
        "jogo_id": jogo_id,
        "apelido": apelido,
        "valor_ganho": valor,
        "tempo_segundos": tempo,
        "acertos": acertos,
        "motivo_encerramento": motivo,
        "data_hora": datetime.now(timezone.utc).isoformat(),
    })

def consultar_ranking():
    try:
        jogo_id = buscar_jogo_id()
        if jogo_id is None:
            print(f"O jogo '{NOME_JOGO}' não existe no banco.")
            return
        resposta = requisitar("GET", "ranking", params={
            "select": "posicao,apelido,valor_ganho,tempo_segundos,data_hora",
            "jogo_id": f"eq.{jogo_id}",
            "posicao": "lte.10",
            "order": "posicao.asc",
        })
    except ErroBanco as erro:
        print(f"Não foi possível consultar o ranking: {erro}")
        return
    linhas = resposta.json()
    if not linhas:
        print("Ainda não há partidas no ranking.")
        return
    print(f"\n===== RANKING - {NOME_JOGO} =====")
    print(f"{'Pos':<4}{'Apelido':<22}{'Valor':<18}{'Tempo':<12}Data")
    for item in linhas:
        print(f"{item['posicao']:<4}{item['apelido']:<22}"
              f"{formatar_dinheiro(item['valor_ganho']):<18}"
              f"{formatar_tempo(item['tempo_segundos']):<12}{formatar_data(item['data_hora'])}")

def menu():
    while True:
        print("\n=== SHOW DO MILHÃO - PYTHON ===")
        print("1) Iniciar partida")
        print("2) Consultar ranking")
        print("0) Sair")
        opcao = input("Escolha: ").strip()
        if opcao == "1":
            jogar()
        elif opcao == "2":
            consultar_ranking()
        elif opcao == "0":
            print("Até a próxima!")
            break
        else:
            print("Opção inválida.")

In [ ]:
# 4. Executar o Menu do Jogo
menu()